# Building GPT

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# getting the input.

words = open("input.txt").read()

In [ ]:
# now, we build the vocabulary.
# vocabulary is nothing but how many unique characters are there in the input text. we will use this to encode the text into integers.

vocab = sorted(list(set(words)))
vocab_size = len(vocab)
print(''.join(vocab))

In [ ]:
print(vocab_size)

In [ ]:
# we're creating a mapping from char to int, and vice-versa.

stoi = { ch:i for i,ch in enumerate(vocab) }
itos = { i:ch for i,ch in enumerate(vocab) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a strin

## Making our dataset now.

### We're going to be splitting this into train and val sets.
### train - 90%, val - 10%.

In [ ]:
data = torch.tensor(encode(words), dtype=torch.long)
print(data.shape, data.dtype)

In [ ]:
# data[:1000]

In [ ]:
# now that we have the data in the form of a tensor
# we want to split this into train and val sets.

n = int(0.9*len(data))

train_data = data[:n]
val_data = data[n:]

In [ ]:
# get_batch basically just provides a single batch of inputs to the network.
# it creates the inputs, and also the targets against which the inputs will be evaluated (loss).

torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

# print('----')

# for b in range(batch_size): # batch dimension
#     for t in range(block_size): # time dimension
#         context = xb[b, :t+1]
#         target = yb[b,t]
#         print(f"when input is {context.tolist()} the target: {target}")

In [ ]:
n_embed = 32

token_embedding_table = nn.Embedding(vocab_size, n_embed)
lm_head = nn.Linear(n_embed, vocab_size)

logits = token_embedding_table(xb)
print("logits 1 shape is : ", logits.shape)

logits = lm_head(logits)
print("logits 2 shape is : ", logits.shape)


In [41]:
# now, we're going to build up an embedding lookup table.
# this is basically a giant table that contains the embedding vectors for each character in the vocab.
# so, this is going to be of shape (65, 32).
# 65 because we have 65 unique chars in the vocab, and 32 as the embedding dimension because we want each char to be represented as a 32-dimensional vector.

# we also want to add positional embeddings, which will help the model understand the order of the characters in the input sequence.
# because, the order matters too. "cat in a hat" is different from "hat in a cat".
# and how are these gotten and represented?
# similar to the token embeddings, we can have a posi embedding block of tunable parameters, of shape (block_size, n_embed).
# so, logits (without batches) are going to be (T, n_embed), and the posi embeddings are going to be (T, n_embed) as well, so we can just add them together.


n_embed = 32


class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # (65, 32)
        self.lm_head = nn.Linear(n_embed, vocab_size) # (32, 65)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B, T, 32)
        logits = self.lm_head(logits) # (B, T, 65)

        # now, we want to do a forward pass, which is just calculating the logits, and the loss
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            print("logits.shape is: ", logits.shape)
            print("targets.shape is: ", targets.shape)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [ ]:
m = BigramLanguageModel()

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
# the initial loss, before any backprop even happens is 4.6695.
# now, let's automate the forward & backward passes.


batch_size = 32

for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    print(f"Step {steps}, Loss: {loss.item()}")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()